In [18]:
from copy import deepcopy

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt

In [19]:
### Utilities

def deep_update(base: dict, updates: dict) -> dict:
    """Recursively update dicts (for cfg overriding)"""
    out = deepcopy(base)
    for k, v in updates.items():
        if isinstance(v, dict) and isinstance(out.get(k, None), dict):
            out[k] = deep_update(out[k], v)
        else:
            out[k] = deepcopy(v)
    return out


def set_cpu_safety(num_threads: int = 1):
    """Helps me kernel not die"""
    torch.set_num_threads(num_threads)
    torch.set_num_interop_threads(num_threads)

In [20]:
### Synthetic spectra generation

DEFAULT_GEN = dict(
    ### K distribution:
    k_mode="poisson",          ### "poisson" or "uniform"
    p_zero=0.0,
    k_mean=3.0,
    k_tail_prob=0.50,
    k_tail_min=6,

    ### Component priors:
    amp_lognorm_mu=0.0,
    amp_lognorm_sigma=1.0,
    sigma_min=0.1,
    sigma_max=10.0,

    ### Optional blending (to mess with things even more)
    blend_cluster_prob=0.0,
    cluster_width_range=(1.0, 30.0),

    ### Baseline/noise:
    noise_std_range=(0.02, 0.15),
    baseline_poly_prob=0.5,
    baseline_max_slope=0.02,
    baseline_max_quad=0.0002,
)


def _make_v_axis(cfg: dict) -> np.ndarray:
    vmin, vmax = cfg["vrange"]
    C = int(cfg["n_channels"])
    return np.linspace(vmin, vmax, C, dtype=np.float32)


def generate_spectrum(cfg: dict, rng: np.random.Generator, v_axis=None) -> dict:
    """
    Returns dict with stable keys used across notebooks:
      spec       : noisy spectrum, (C,)
      spec_clean : pure Gaussian spectrum (no baseline/noise), (C,)
      k          : int, number of sampled components
      noise_std  : (1,)
      v_axis     : (C,)
    """
    vmin, vmax = cfg["vrange"]
    C = int(cfg["n_channels"])
    Kmax = int(cfg["max_components"])
    min_components = int(cfg.get("min_components", 0))

    gen = deep_update(DEFAULT_GEN, cfg.get("gen", {}))
    k_tail_max = min(10, Kmax)

    v = v_axis if v_axis is not None else _make_v_axis(cfg)

    ### sample k
    if rng.random() < gen["p_zero"]:
        k = 0
    else:
        if gen.get("k_mode", "poisson") == "uniform":
            ### uniform over 0..Kmax (or 1..Kmax if you want to avoid empties)
            k = int(rng.integers(0, Kmax + 1))
            if k > 0:
                k = max(k, min_components)
        else:
            ### original poisson + optional tail
            if rng.random() < gen["k_tail_prob"]:
                k = int(rng.integers(gen["k_tail_min"], k_tail_max + 1))
            else:
                k = max(1, int(rng.poisson(gen["k_mean"])))
            k = max(k, min_components) if k > 0 else 0
    
    k = min(k, Kmax)

    ### allocate component arrays
    A = np.zeros(Kmax, dtype=np.float32)
    mu = np.zeros(Kmax, dtype=np.float32)
    sig = np.ones(Kmax, dtype=np.float32)

    if k > 0:
        ### centers
        if rng.random() < gen["blend_cluster_prob"] and k >= 2:
            center = rng.uniform(vmin + 0.2*(vmax - vmin), vmax - 0.2*(vmax - vmin))
            cw = rng.uniform(*gen["cluster_width_range"])
            mus = center + rng.normal(0.0, cw, size=k)
            mus = np.clip(mus, vmin, vmax)
        else:
            mus = rng.uniform(vmin, vmax, size=k)

        ### widths + amps
        sigs = rng.uniform(gen["sigma_min"], gen["sigma_max"], size=k)
        amps = rng.lognormal(mean=gen["amp_lognorm_mu"], sigma=gen["amp_lognorm_sigma"], size=k)
        amps = amps / (np.percentile(amps, 90) + 1e-6)  ### stabilize scale

        ### sort by centroid
        order = np.argsort(mus)
        mus, sigs, amps = mus[order], sigs[order], amps[order]

        A[:k] = amps.astype(np.float32)
        mu[:k] = mus.astype(np.float32)
        sig[:k] = sigs.astype(np.float32)

    ### build clean spectrum
    spec = np.zeros(C, dtype=np.float32)
    for i in range(k):
        dv = (v - mu[i]) / (sig[i] + 1e-6)
        spec += A[i] * np.exp(-0.5 * dv * dv).astype(np.float32)
    spec_clean = spec.copy()

    ### baseline
    if rng.random() < gen["baseline_poly_prob"]:
        x = np.linspace(-1, 1, C, dtype=np.float32)
        slope = rng.uniform(-gen["baseline_max_slope"], gen["baseline_max_slope"])
        quad = rng.uniform(-gen["baseline_max_quad"], gen["baseline_max_quad"])
        spec += (slope * x + quad * (x**2)).astype(np.float32)

    ### noise
    noise_std = float(rng.uniform(*gen["noise_std_range"]))
    spec += rng.normal(0.0, noise_std, size=C).astype(np.float32)

    return dict(
        spec=spec,
        spec_clean=spec_clean,
        k=k,
        noise_std=np.array([noise_std], dtype=np.float32),
        v_axis=v,
    )

In [21]:
### Dataset generation

class SyntheticSpectraDataset(Dataset):
    """
    Deterministic synthetic dataset indexed by base_seed + idx.

    Returned keys:
      spec   : (C,) float
      K_true : (1,) long, integer class in [0, Kmax]
    """
    def __init__(self, cfg: dict, n_samples: int, base_seed: int = 0):
        self.cfg = cfg
        self.v_axis = _make_v_axis(cfg)  ### (C,)
        self.n_samples = int(n_samples)
        self.base_seed = int(base_seed)

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx: int):
        rng = np.random.default_rng(self.base_seed + int(idx))
        ex = generate_spectrum(self.cfg, rng=rng, v_axis=self.v_axis)

        K_true = np.array([ex["k"]], dtype=np.int64)  ### (1,)

        return {
            "spec": torch.from_numpy(ex["spec"]).float(),
            "K_true": torch.from_numpy(K_true).long(),  ### (1,)
            ### kept only for sanity plotting
            "spec_clean": torch.from_numpy(ex["spec_clean"]).float(),
        }


BASE_CFG = dict(
    n_channels=256,
    min_components=0,
    max_components=10,
    vrange=(-200.0, 200.0),
    gen=deepcopy(DEFAULT_GEN),
)


def make_loaders(cfg: dict, *, n_train=50_000, n_val=5_000, bs_train=128, bs_val=256):
    train_ds = SyntheticSpectraDataset(cfg, n_samples=n_train, base_seed=0)
    val_ds   = SyntheticSpectraDataset(cfg, n_samples=n_val, base_seed=10_000_000)

    train_loader = DataLoader(train_ds, batch_size=bs_train, shuffle=True, num_workers=0, pin_memory=False)
    val_loader   = DataLoader(val_ds,   batch_size=bs_val,   shuffle=False, num_workers=0, pin_memory=False)
    return train_loader, val_loader


In [22]:

@torch.no_grad()
def plot_example(ds: Dataset, idx: int = 0, title=""):
    ex = ds[idx]
    v = ds.v_axis
    plt.figure(figsize=(10, 3))
    plt.plot(v, ex["spec"].numpy(), label="spec (noisy)", lw=1.8, alpha=0.8)
    plt.plot(v, ex["spec_clean"].numpy(), label="spec_clean", lw=1.8, alpha=0.8)
    plt.title(title or f"Example index={idx}  K_true={int(ex['K_true'].item())}")
    plt.xlabel("Velocity (km/s)")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [23]:
### Model for scheme C

class CountNet1D_Classify(nn.Module):
    def __init__(self, Kmax: int, width: int = 64):
        super().__init__()
        self.Kmax = int(Kmax)
        self.conv = nn.Sequential(
            nn.Conv1d(1, width, 9, padding=4),
            nn.ReLU(),
            nn.Conv1d(width, width, 9, padding=4),
            nn.ReLU(),
            nn.Conv1d(width, width, 9, padding=4),
            nn.ReLU(),
        )
        self.head = nn.Linear(width, self.Kmax + 1)  ### logits for classes 0..Kmax

    def forward(self, x):
        '''x: (B, C)'''
        h = self.conv(x.unsqueeze(1))      ### (B, width, C)
        h = h.mean(dim=-1)                 ### (B, width)
        logits = self.head(h)              ### (B, Kmax+1)
        return logits

In [24]:
### Training loop

def train_count_classify(model, train_loader, val_loader, *, device="cpu", lr=1e-3, epochs=3, log_every=200):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    for ep in range(1, epochs + 1):
        model.train()
        running = 0.0

        for step, batch in enumerate(train_loader, start=1):
            x = batch["spec"].to(device)                         ### (B,C)

            ### normalization (same as Scheme B)
            x = x - x.mean(dim=1, keepdim=True)
            x = x / (x.std(dim=1, keepdim=True) + 1e-6)

            K = batch["K_true"].to(device).long().squeeze(-1)    ### (B,)

            logits = model(x)                                    ### (B, Kmax+1)
            loss = loss_fn(logits, K)

            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

            running += loss.item()
            if step % log_every == 0:
                print(f"epoch {ep} step {step} train_loss {running/log_every:.4f}")
                running = 0.0

        ### validation: accuracy + MAE (argmax) + MAE (expected K)
        model.eval()
        n = 0
        n_correct = 0
        mae_argmax = 0.0
        mae_expected = 0.0

        with torch.no_grad():
            for batch in val_loader:
                x = batch["spec"].to(device)
                x = x - x.mean(dim=1, keepdim=True)
                x = x / (x.std(dim=1, keepdim=True) + 1e-6)

                K = batch["K_true"].to(device).long().squeeze(-1)    ### (B,)
                logits = model(x)

                Kp = torch.argmax(logits, dim=1)                     ### (B,)
                probs = torch.softmax(logits, dim=1)                 ### (B, Kmax+1)

                ks = torch.arange(probs.shape[1], device=device).float()
                Ke = (probs * ks[None, :]).sum(dim=1)                ### (B,)

                n += x.size(0)
                n_correct += (Kp == K).sum().item()
                mae_argmax += (Kp - K).abs().sum().item()
                mae_expected += (Ke - K.float()).abs().sum().item()

        acc = n_correct / max(1, n)
        print(f"epoch {ep} val_acc {acc:.3f}  val_K_MAE(argmax) {mae_argmax/n:.3f}  val_K_MAE(E[K]) {mae_expected/n:.3f}")

    return model

In [25]:
### Collect predictions

@torch.no_grad()
def collect_count_predictions(model, loader, *, device="cpu", Kmax=10, max_batches=50):
    model.eval()
    y_true = []
    y_pred = []
    y_exp = []

    for bi, batch in enumerate(loader):
        if bi >= max_batches:
            break

        x = batch["spec"].to(device)
        x = x - x.mean(dim=1, keepdim=True)
        x = x / (x.std(dim=1, keepdim=True) + 1e-6)

        K = batch["K_true"].to(device).long().squeeze(-1)

        logits = model(x)                         ### (B, Kmax+1)
        probs = torch.softmax(logits, dim=1)

        Kp = torch.argmax(logits, dim=1).long()   ### (B,)

        ks = torch.arange(Kmax + 1, device=device).float()
        Ke = (probs * ks[None, :]).sum(dim=1)      ### (B,)

        y_true.append(K.cpu().numpy())
        y_pred.append(Kp.cpu().numpy())
        y_exp.append(Ke.cpu().numpy())

    return np.concatenate(y_true), np.concatenate(y_pred), np.concatenate(y_exp)

In [26]:
### Run an experiment
### Note: If kernel crashes from CPU oversubscription, run set_cpu_safety(1) in the first cell before any other code.

cfg = deep_update(BASE_CFG, dict(
    gen=dict(
        k_mode="uniform",
        p_zero=0.0,        ### optional: if you want true uniform, keep 0.0 and let uniform include 0
        k_tail_prob=0.0,   ### irrelevant in uniform mode
    )
))
Kmax = int(cfg["max_components"])

train_loader, val_loader = make_loaders(cfg, n_train=50_000, n_val=5_000, bs_train=128, bs_val=256)

device = "cuda" if torch.cuda.is_available() else "cpu"

model = CountNet1D_Classify(Kmax=Kmax, width=64)

### optional: sanity plot
plot_example(val_loader.dataset, idx=0)

model = train_count_classify(
    model,
    train_loader,
    val_loader,
    device=device,
    lr=1e-3,
    epochs=3,
    log_every=200,
)

RuntimeError: Error: cannot set number of interop threads after parallel work has started or set_num_interop_threads called

In [17]:
### Diagnostics (same set as Scheme B)

y_true, y_pred, y_exp = collect_count_predictions(model, val_loader, device=device, Kmax=Kmax, max_batches=50)

### Choose which prediction to visualize
###   y_pred : argmax class
###   round(E[K]) : rounded expected value
y_show = y_pred
### y_show = np.clip(np.rint(y_exp), 0, Kmax).astype(int)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

### (1) 2D histogram / confusion-style heatmap
edges = np.arange(-0.5, Kmax + 1.5, 1.0)
H, xedges, yedges = np.histogram2d(y_true, y_show, bins=[edges, edges])

im = axes[0].imshow(
    H.T,
    origin="lower",
    aspect="equal",
    interpolation="nearest",
    extent=[edges[0], edges[-1], edges[0], edges[-1]],
)
#axes[0].plot([0, Kmax], [0, Kmax], lw=1.5, color="k")
axes[0].set_xlabel("K true")
axes[0].set_ylabel("K pred")
axes[0].set_title("True vs Pred (2D hist)")
ticks = np.arange(0, Kmax + 1, 1)
axes[0].set_xticks(ticks)
axes[0].set_yticks(ticks)
fig.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04, label="count")

### (2) Row-normalized view: P(K_pred | K_true)
Hn = H / (H.sum(axis=1, keepdims=True) + 1e-12)
im2 = axes[1].imshow(
    Hn.T,
    origin="lower",
    aspect="equal",
    interpolation="nearest",
    extent=[edges[0], edges[-1], edges[0], edges[-1]],
)
#axes[1].plot([0, Kmax], [0, Kmax], lw=1.5, color="k")
axes[1].set_xlabel("K true")
axes[1].set_ylabel("K pred")
axes[1].set_title("True vs Pred (row-normalized)")
axes[1].set_xticks(ticks)
axes[1].set_yticks(ticks)
fig.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04, label="P(K_pred | K_true)")

### (3) Error histogram
err = (y_show - y_true)
bins = np.arange(err.min() - 0.5, err.max() + 1.5, 1.0)
axes[2].hist(err, bins=bins, color="cornflowerblue", alpha=0.5)
axes[2].hist(err, bins=bins, histtype="step", color="k", lw=2)
axes[2].set_xlabel("K_pred - K_true")
axes[2].set_title("Count error histogram")

plt.tight_layout()
plt.show()
plt.close(fig)



KeyError: 'k_tail_min'

In [ ]:
### MAE vs K_true
mae_by_k = []
ks = np.arange(0, Kmax + 1)
for k in ks:
    m = (y_true == k)
    mae_by_k.append(np.mean(np.abs(y_show[m] - y_true[m])) if np.any(m) else np.nan)

plt.figure(figsize=(6, 3))
plt.plot(ks, mae_by_k, marker="o")
plt.xlabel("K_true")
plt.ylabel("MAE")
plt.title("MAE vs K_true")
plt.tight_layout()
plt.show()

### K_true frequency
counts = np.array([(y_true == k).sum() for k in ks])

plt.figure(figsize=(6, 3))
plt.bar(ks, counts)
plt.xlabel("K_true")
plt.ylabel("count")
plt.title("K_true frequency in eval sample")
plt.tight_layout()
plt.show()